# Day 04 · Resource 기반 MCP 설계 실습
읽기 전용 데이터를 **정적 → 동적(템플릿) → 목록 → 파일 → CSV** 리소스로 안전하게 노출합니다. in-process Client로 검증.


## 0 · 설치 & 데이터 준비


In [ ]:
%pip install -q fastmcp


In [ ]:
# 예제 파일·CSV 만들기
import os, csv
os.makedirs("docs", exist_ok=True)
open("docs/leave.txt", "w", encoding="utf-8").write("연차는 3영업일 전 신청·팀장 승인")
open("docs/expense.txt", "w", encoding="utf-8").write("비용은 영수증 첨부 후 재무팀 최종 승인")
with open("employees.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f); w.writerow(["id","name","dept"])
    w.writerow(["1001","김민수","개발"]); w.writerow(["1002","이서연","디자인"])
print("준비 완료:", os.listdir("docs"))


## Lab 1 · 정적 리소스
고정 URI. description을 꼭 명시.


In [ ]:
from fastmcp import FastMCP, Client
mcp = FastMCP("ResourceServer")

@mcp.resource("info://about", description="이 서버 소개")
def about() -> str:
    return "사내 지식·정책을 읽기 전용으로 제공하는 서버"

async with Client(mcp) as c:
    print((await c.read_resource("info://about"))[0].text)


## Lab 2 · 동적(템플릿) 리소스
`{topic}`이 함수 인자로 들어옵니다 (REST 경로 파라미터처럼).


In [ ]:
POLICY = {"leave": "연차는 3영업일 전 신청·승인",
          "expense": "비용은 영수증 첨부 후 재무팀 최종 승인"}

@mcp.resource("policy://{topic}", description="주제별 사내 정책을 반환")
def policy(topic: str) -> str:
    return POLICY.get(topic, "해당 정책이 없습니다.")

async with Client(mcp) as c:
    for t in ["leave", "expense", "unknown"]:
        print(t, "→", (await c.read_resource(f"policy://{t}"))[0].text)


## Lab 3 · 목록(인덱스) 리소스
무엇을 읽을 수 있는지 알려 발견 가능성을 높입니다.


In [ ]:
import json

@mcp.resource("policy://index", description="이용 가능한 정책 주제 목록")
def policy_index() -> list:
    return list(POLICY.keys())

async with Client(mcp) as c:
    print("목록:", (await c.read_resource("policy://index"))[0].text)


## Lab 4 · 폴더 파일을 안전하게 노출
경로 매개변수는 **반드시 검증**(../ 탈출 방지).

> ⚠️ URI 스킴 주의: `file://`은 표준상 `//` 뒤를 **호스트**로 해석해 끝에 `/`가 붙어 템플릿 매칭이 깨집니다. 그래서 커스텀 스킴 **`docs://{name}`** 을 씁니다(다른 리소스 `policy://`·`emp://` 와 일관).

In [ ]:
from pathlib import Path
BASE = Path("docs").resolve()

# 주의: file:// 스킴은 URI 표준상 뒤를 '호스트'로 해석해 끝에 /가 붙어
# 템플릿 {name} 매칭이 깨집니다. 다른 리소스(policy://, emp://)처럼
# 커스텀 스킴 docs:// 를 사용합니다.
@mcp.resource("docs://{name}", description="docs 폴더의 텍스트 파일을 읽는다")
def read_doc(name: str) -> str:
    p = (BASE / name).resolve()
    if BASE not in p.parents:
        raise ValueError("허용되지 않은 경로")
    return p.read_text(encoding="utf-8")

async with Client(mcp) as c:
    print((await c.read_resource("docs://leave.txt"))[0].text)

## Lab 5 · CSV 백엔드 리소스
사번으로 직원 한 행을 반환(읽기 전용).


In [ ]:
import csv
ROWS = {r["id"]: r for r in csv.DictReader(open("employees.csv", encoding="utf-8"))}

@mcp.resource("emp://{eid}", description="사번으로 직원 정보를 반환")
def employee(eid: str) -> dict:
    return ROWS.get(eid, {"error": "없는 사번"})

async with Client(mcp) as c:
    print((await c.read_resource("emp://1001"))[0].text)


## Lab 6 · 형식(MIME)을 밝혀 노출하기
같은 반환값도 **`mime_type`**을 지정하면 클라이언트가 알맞게 처리합니다 (슬라이드: MIME·바이너리).
- dict → `application/json`, 마크다운 → `text/markdown`, 이미지·PDF → **바이너리(blob)**
- 안 밝히면 텍스트로 취급돼 이미지·구조가 깨질 수 있습니다.
> 직접 해보기: `fmt://sales` 를 `text/csv` 로 노출해 보세요.


In [ ]:
# mime_type 으로 형식을 명확히 — 텍스트/JSON은 text, 바이너리는 blob 으로 돌아온다
@mcp.resource("fmt://config", mime_type="application/json", description="설정을 JSON으로")
def fmt_config() -> dict:
    return {"version": "1.0", "env": "dev"}

@mcp.resource("fmt://readme", mime_type="text/markdown", description="위키를 마크다운으로")
def fmt_readme() -> str:
    return "# 사내 위키\n- 연차는 3영업일 전 신청"

# 이미지·PDF는 bytes 로 반환하면 base64 blob 이 된다 (아래는 1x1 PNG 예시)
LOGO_PNG = bytes.fromhex("89504e470d0a1a0a0000000d49484452000000010000000108060000001f15c489"
                         "0000000d4944415478da63f8cfc0f01f0005000155e2e5b90000000049454e44ae426082")
@mcp.resource("fmt://logo", mime_type="image/png", description="로고 이미지(바이너리)")
def fmt_logo() -> bytes:
    return LOGO_PNG

async with Client(mcp) as c:
    for uri in ["fmt://config", "fmt://readme", "fmt://logo"]:
        item = (await c.read_resource(uri))[0]
        kind = "text" if getattr(item, "text", None) is not None else "blob(바이너리)"
        print(f"{uri:14} mime={item.mimeType:18} -> {kind}")


## Lab 7 · 도전 (선택)
- **A** 같은 데이터에 조회=Resource, 수정=Tool 을 함께 제공
- **B** 실제 이미지·PDF 파일을 읽어 알맞은 `mime_type`으로 노출 (Lab 6 확장)
- **C** Day 01~02의 청크를 `chunk://{id}` 리소스로 노출해 '지식 서버' 뼈대 만들기

> 📝 **산출물**: 파일·CSV를 읽기 전용으로 안전하게 노출하는 Resource 중심 서버.
